In [449]:
%reset -f
from build123d import *
from ocp_vscode import *
import cadquery as cq
import time
import math
from library.tools import *
import sys

ALL UNITS IN MM
DON'T @ ME

In [450]:
%reload_ext ocp_vscode
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [451]:

print(f"Python executable: {sys.executable}")
print(f"OCP-vscode location: {sys.modules.get('ocp_vscode', 'Not found')}")

Python executable: c:\Users\Kaoti\parthenon\.venv\Scripts\python.exe
OCP-vscode location: <module 'ocp_vscode' from 'c:\\Users\\Kaoti\\parthenon\\.venv\\Lib\\site-packages\\ocp_vscode\\__init__.py'>


Toy parts: 

In [452]:
reset_show()
box = cq.Workplane().box(1,2,1).edges().chamfer(0.4)
show_object(box, name="Chamfered Box", options={"alpha": 0.8})

c


In [453]:
sphere = cq.Workplane().sphere(0.6)
reset_show()
show_object(sphere)

c


In [454]:
reset_show()
box = Box(1, 2, 1)
show(box, reset_camera=Camera.RESET)
print(f"Box created: {box}")  # Should show object details
# Does anything appear in the viewer panel?

c
Box created: Box at 0x1531002d710, label(), #children(0)


Pseudocode: 

Construct primitives (Initial GT frame circle,)
Orient primitives
Translate primitives
Fuse primitives



Actual code: 

In [455]:

reset_show()

.translate(x,y,z)
reset_camera=Camera.RESET
add(mypart)
reset_show
Cylinder(radius, height)

In [456]:
# Variables for basis cylinder

# Cylinder subtraction
chamberRadius1=27.5/2 
chamberRadius2=30.150/2 - 0.025 #0.025mm thicken step in arbor npx backbone. built in here to the radius
chamberRadius3=36/2
chamberGripOuterHeight=10
chamberGripInnerHeight=5

# Cutaway triangle
cutTrianglePrimaryAngle=123.547500 # This is the cutout angle in the basis cylinder
cutTriangleSecondaryAngle=88.546500 # This is the cutout angle relative to the topmost throughhole axis
cutTriangleHeight=4
triangleAngle = 123
triangleThickness = 4
vertices1 = (0,0,0)
vertices2 = (-2*chamberRadius3,0,0)
vertices3 = (-3*chamberRadius3*(math.cos(math.radians(triangleAngle))), -3*chamberRadius3*(math.sin(math.radians(triangleAngle))),0)

# Throughhole parameters
throughHoleAngle=90.0 # This is the angle between the throughholes
throughHoleRadius=2.500/2 # This is the radius of the throughholes

# Small additive directional indicator triangle
indicatorTriangleAngle = -1 * (33.456000 + throughHoleAngle)
indicatorTriangleSideLength = 3.46410
indicatorTriangleInternalAngles = 180/3
indicatorOffset = 3.0 / 2
indicatorHorizontalDisplacement = 14.50000
indicatorTotalYDisplacement = indicatorHorizontalDisplacement + indicatorOffset + (chamberRadius3 - chamberRadius1)/2
indicatorTriangleThickness = 0.50000


In [457]:
# Basis cylinder

# Cylinder parts
cylinderTranslationVector=(0,0,(chamberGripOuterHeight-chamberGripInnerHeight)/2)

cylinder1 = Cylinder(radius=chamberRadius1, height=chamberGripOuterHeight)
cylinder2 = Cylinder(radius=chamberRadius2, height=chamberGripInnerHeight).translate(cylinderTranslationVector)
cylinder3 = Cylinder(radius=chamberRadius3, height=chamberGripOuterHeight)

# Throughholes
throughHole1TranslationVector=(0,(chamberRadius2+chamberRadius3)/2,1.15100+throughHoleRadius,0)
throughHole1RotationVector=(1,0,0)

throughHole2TranslationVector=((chamberRadius2+chamberRadius3)/2,0,1.15100+throughHoleRadius)
throughHole2RotationVector=(0,1,0)

throughHole1Cylinder = Cylinder(radius=throughHoleRadius, height=chamberRadius1/2).rotate(
    axis = Axis(Vector(0,0,0), throughHole1RotationVector),
    angle=90
).translate(throughHole1TranslationVector)

throughHole2Cylinder = Cylinder(radius=throughHoleRadius,height=chamberRadius1/2).rotate(
    axis = Axis(Vector(0,0,0), throughHole2RotationVector),
    angle = 90
).translate(throughHole2TranslationVector)

# Cutaway Triangle
with BuildPart() as bp:
    with BuildSketch():
        Polygon(vertices1, vertices2, vertices3, align=None)
    extrude(amount=triangleThickness)

basisTriangle = bp.part.translate((0,0,(chamberGripOuterHeight/2 - triangleThickness)))

# Directional Indicator Triangle
with BuildPart() as bp:
    with BuildSketch():
        Polygon((-indicatorOffset,0,0), (indicatorOffset,0,0), (0,indicatorOffset*2,0), align=None)
    extrude(amount=indicatorTriangleThickness)

indicatorTriangleTranslationVector = (0,indicatorHorizontalDisplacement,-(indicatorTriangleThickness + chamberGripOuterHeight)/2)
indicatorTriangleRotationVector = (0,0,1)

indicatorTriangle = bp.part.translate(indicatorTriangleTranslationVector).rotate(
    axis = Axis(Vector(0,0,0), indicatorTriangleRotationVector),
    angle = indicatorTriangleAngle
)



For subtracting items as masks:     add(cylinder1, mode=Mode.SUBTRACT)

In [458]:
# Building the whole basis cylinder

reset_show()

# Doing the initial construction of the basis cylinder
with BuildPart() as basisCylinder:
    add(cylinder3)
    add(cylinder1, mode=Mode.SUBTRACT)
    add(cylinder2, mode=Mode.SUBTRACT)
    add(throughHole1Cylinder, mode=Mode.SUBTRACT)
    add(throughHole2Cylinder, mode=Mode.SUBTRACT)
    add(basisTriangle, mode=Mode.SUBTRACT)
    add(indicatorTriangle)

# Flipping the cylinder, since I decided building it upside-down at an odd angle was a great idea
basisCylinderRotationVector = (1,0,0)
basisCylinder = basisCylinder.part.rotate(
    axis = Axis(Vector(0,0,0), basisCylinderRotationVector),
    angle = 180
)

# Rotating the cylinder so the indicator triangle points to +y
basisCylinderRotationVector = (0,0,1)
basisCylinder = basisCylinder.rotate(
    axis = Axis(Vector(0,0,0), basisCylinderRotationVector),
    angle = 180 - (33.456000 + throughHoleAngle) 
)

# Translating the cylinder down 5mm in y axis so it aligns correctly with the chamber mesh. See note in next code box.
basisCylinderTranslationVector = (0,0,-5)
basisCylinder = basisCylinder.translate(basisCylinderTranslationVector)

show(basisCylinder, reset_camera=Camera.RESET)

c


In [459]:
# Importing reference chamber mesh - Goliath Posterior V3 starting V3 
chamberMold = Mesher().read("chamberMoldMeshGoliathPosteriorV3.stl")[0]

# NOTE: MESHES CANNOT BE TRANSLATED. TRANSLATE OTHER OBJECTS AROUND THEM.

# Displaying the mesh and the cylinder both
reset_show()
show(chamberMold,basisCylinder,reset_camera=Camera.RESET)

cc


Immediately below this point: Probably need to integrate coordinates from SPOTS somehow. Likely, also need a least-squares fit line for the arbor, and some parameter permutation if we want a normal offset from the arbor line. 

In [460]:
# Retrieving / generating points for probe penetration

# Parameters
meshZ = -5
GTHoleDiameter = 0.675
recessDiameter = 3.1
GTExteriorDiameter = 2

# Penetration center points
penetration1 = (0,7.7, meshZ)
penetration2 = (0,2.475,meshZ)
penetration3 = (0,-1.775,meshZ)
penetration4 = (0,-7,meshZ)
points = (penetration1, penetration2, penetration3, penetration4)
circularLocations = []

# Alternative construction which leaves each point accessible
for x in points:
    with BuildSketch() as sk:
            with Locations(x):
                Circle(radius = GTExteriorDiameter, align=None)
    circularLocations.append(sk)
print(circularLocations)

# Everybody gets a diameter
with BuildSketch() as sk:
    with Locations(points):
        Circle(radius = 2)
        Circle(radius = GTHoleDiameter/2, mode = Mode.SUBTRACT)

# These sketches now exist
penetrationLocations = sk

# Displaying everything at once
show (chamberMold, basisCylinder, circularLocations, reset_camera = Camera.RESET)

[<build123d.build_sketch.BuildSketch object at 0x00000152F9ECC950>, <build123d.build_sketch.BuildSketch object at 0x00000152F9ECD790>, <build123d.build_sketch.BuildSketch object at 0x00000152F9ECFE90>, <build123d.build_sketch.BuildSketch object at 0x00000152F9ECD490>]
----cccccc


In [461]:
# Mesh projected curves
projectedCurves = []
for x in circularLocations:
    wire = x.sketch.faces()[0].outer_wire()
    hits = wire.project_to_shape(chamberMold,direction=(0,0,-1))
    projectedCurves.append(hits)

print(projectedCurves)

show (chamberMold, basisCylinder, circularLocations, projectedCurves, reset_camera = Camera.RESET)

[[<build123d.topology.one_d.Wire object at 0x00000152E70624D0>, <build123d.topology.one_d.Wire object at 0x00000152E70E3CD0>], [<build123d.topology.one_d.Wire object at 0x00000152E7062DD0>, <build123d.topology.one_d.Wire object at 0x00000152E7061350>], [<build123d.topology.one_d.Wire object at 0x00000152E7063950>, <build123d.topology.one_d.Wire object at 0x00000152E7062F50>], [<build123d.topology.one_d.Wire object at 0x00000152E70614D0>, <build123d.topology.one_d.Wire object at 0x00000152E70621D0>]]
----cccccc


In [462]:
# Building surfaces from projected curves

frameFaces = []

for x in circularLocations:
    face = x.sketch.faces()[0]
    center = face.center()
    axis = Axis((center.X, center.Y, chamberMold.bounding_box().max.Z), (0,0,-1))
    point, normal = chamberMold.find_intersection_points(axis)[-1]
    tiltedPlane = Plane(origin=point, z_dir = normal)
    frameFaces.append(face.located(Location(tiltedPlane)))

In [463]:
# Does it work?

show (frameFaces, basisCylinder, circularLocations, projectedCurves, reset_camera = Camera.RESET)

----ccccccccc


In [464]:
# Offsetting the projected curves to provide targets for extrusion limits

offsetFrameFaces = []
zPlanarOffset = 2   

for x in frameFaces:
    plane1 = Plane(x)
    offsetPlane = Plane(origin = plane1.origin + (0,0,zPlanarOffset), x_dir=plane1.x_dir, z_dir = plane1.z_dir)
    offsetFrameFaces.append(offsetPlane)

show(offsetFrameFaces, basisCylinder, circularLocations, projectedCurves, reset_camera = Camera.RESET)


----ccccc


In [465]:
# Ok, now to transform the faces which the shafts start extruding on into planes so they can be useful
startingOffsetPlanes = []
shaftHeight = 3.85 # This is a rough estimation based on Anna's onshape
shaftDiameter = 4
shaftRadius = shaftDiameter/2
innerShaftDiameter = 0.67500
innerShaftRadius = innerShaftDiameter/2

for x in offsetFrameFaces: # These are actually planes, not faces
    appendMe = Plane(origin = x.origin + (0,0,shaftHeight), x_dir = (1,0,0), z_dir = (0,0,1))
    startingOffsetPlanes.append(appendMe)

In [466]:
# Generating solids from bottom surface planes for later use
bottomSurfaceSolids = []

for x in offsetFrameFaces:
    with BuildPart() as cyl:
        with BuildSketch(x):
            Circle(radius = shaftDiameter)
        extrude(amount=1)
    bottomSurfaceSolids.append(cyl.part)

show(bottomSurfaceSolids, startingOffsetPlanes)

cccc


In [467]:
# Building circles on the projected curves
shaftList = []
throughHoles = []
shaftListWithThroughHoles = []
seams = []
j = 0

for i, x in enumerate(startingOffsetPlanes): 
    # First build the outer shaft
    with BuildPart() as shaft:
        with BuildSketch(x) as sk:
            Circle(radius=shaftRadius)
        extrude(until=Until.NEXT, target = bottomSurfaceSolids[i], dir=(0,0,-1))
    shaftList.append(shaft)

    # Next build the throughhole
    with BuildPart() as throughhole:
        with BuildSketch(x) as sk:
            Circle(radius = innerShaftRadius)
        extrude(until = Until.NEXT, target = bottomSurfaceSolids[i], dir=(0,0,-1))
    throughHoles.append(throughhole)

    # Now subtract the throughhole geometry from the shaft geometery
    with BuildPart() as shaftWithThroughHole:
        add(shaftList[i])
        add(throughHoles[i], mode = Mode.SUBTRACT)
    shaftListWithThroughHoles.append(shaftWithThroughHole)

# show the throughholes in red and the shafts in yellow for diagnostic purposes
show(shaftList, throughHoles, colors = ["Yellow", "Red"])


cccccccc


In [468]:
# Fillet slanted bottom edge of shaft and throughhole
targetFaces = []
filletItems = []

for i, x in enumerate(shaftListWithThroughHoles):
    targetEdges = []
    targetFace = min(shaftListWithThroughHoles[i].faces(), key = findZ)
    targetFaces.append(targetFace)
    targetEdge1, targetEdge2 = targetFace.edges()
    targetEdges.append(targetEdge1)
    targetEdges.append(targetEdge2)
    filletItem: Sketch | Part | Curve = fillet(targetEdges, radius = 0.20)
    filletItems.append(filletItem)

shaftListWithThroughHoles = filletItems

show(targetEdges, shaftListWithThroughHoles, reset_camera=Camera.RESET)

cccc


In [469]:
# Fillet-ing top edge of throughholes
filletList = []

for i, x in enumerate(shaftListWithThroughHoles):
    edge = x.edges().filter_by(GeomType.CIRCLE).filter_by(
        lambda a: abs(a.radius - innerShaftRadius) < 1e-6)
    seams.append(edge)

    filletObject = fillet(edge, radius = 0.25)
    filletList.append(filletObject)

show(filletList, reset_camera = Camera.RESET)
shaftListWithThroughHoles = filletList


cccc


In [470]:
# Display everything for the purpose of sanity checks

show(basisCylinder, shaftListWithThroughHoles, chamberMold, alphas=[1.0,1.0,0.8], colors=["#e8b024", "#e8b024", "lightblue"], reset_camera=Camera.RESET)

cccccc


Okay, the actual through holes for the GT have been implanted in the shafts. Location-dependant shaft placement and homing has been completed in a scalable fashion. Now we need to connect the shafts to the basis cylinder. 

In [471]:
# GT Depth Calculator: To-do

In [472]:
# Nubs
# Doing this a little bit differently from Anna's Onshape. Instead of building in the default plane, I'm going to build each set of nubs on the top plane of the actual shaft it's attached to.

parallelNubSeparation = 2.64575
nubLongSide = 3
nubShortSide = 1.35425
nubDepth = 2.5
nubsList = []
rotatorLocations = [1,2,3,4]
rotatorAngle = 90
overlaps = []
nubTemplates = []
nubTemplatesFlattened = []

nubDisplacementVector = (0,parallelNubSeparation/2 + nubShortSide/2,0)
nubRotationVector = (0,0,1)

for i, x in enumerate(startingOffsetPlanes):
    for k, j in enumerate(rotatorLocations):
        nubTemplate = Rectangle(nubLongSide,nubShortSide).located(Location(startingOffsetPlanes[i])).translate(nubDisplacementVector).rotate(
            axis = Axis(startingOffsetPlanes[i].origin, nubRotationVector),
            angle = rotatorAngle*k
        )
        nubTemplates.append(nubTemplate)
        with BuildPart() as nubs:
            extrude(nubTemplate, amount=nubDepth, dir = (0,0,-1))
        nubsList.append(nubs.part)

for i, x in enumerate(nubTemplates):
    nubTemplatesFlattened.append(flattenToXY(nubTemplates[i]))

# Overlap areas between nubs 2,4
overlaps.append(nubTemplatesFlattened[2] & nubTemplatesFlattened[4])

# Overlap areas between nubs 6,8
overlaps.append(nubTemplatesFlattened[6] & nubTemplatesFlattened[8])

# Overlap areas between nubs 8,10
overlaps.append(nubTemplatesFlattened[10] & nubTemplatesFlattened[12])

print(overlaps)

show(basisCylinder, shaftListWithThroughHoles, nubsList, overlaps, reset_camera=Camera.RESET)


[Compound at 0x152e6fea7b0, label(), #children(0), Compound at 0x152e6fea960, label(), #children(0), Compound at 0x152e6feb410, label(), #children(0)]
cccccccccccccccccccccccc


In [473]:
# Constructing overlap solids, pruning union parts between different shafts

overlapSolids = []
prunedParts = []
lofts = []

# Generating overlap structures
for i, x in enumerate(overlaps):
    with BuildPart() as firstpart:
        extrude(overlaps[i], amount = shaftHeight*10, dir = (0,0,-1))
        overlapSolids.append(firstpart.part)

# Deleting overlap between shafts 1 and 2, numbered from +y to -y in global coordinates
with BuildPart() as pt:
    add (nubsList[2])
    add (overlapSolids[0], mode = Mode.SUBTRACT)
    prunedParts.append(pt.part)

with BuildPart() as pt:
    add (nubsList[4])
    add (overlapSolids[0], mode = Mode.SUBTRACT)
    prunedParts.append(pt.part)

loft1 = loftMe(nubsList[2], nubsList[4])

# Deleting overlap between shafts 10 and 12, numbered from +y to -y in global coordinates
with BuildPart() as pt:
    add (nubsList[10])
    add (overlapSolids[2], mode = Mode.SUBTRACT)
    prunedParts.append(pt.part)

with BuildPart() as pt:
    add (nubsList[12])
    add (overlapSolids[2], mode = Mode.SUBTRACT)
    prunedParts.append(pt.part)

loft2 = loftMe(nubsList[10], nubsList[12])

with BuildPart() as pt:
    if startingOffsetPlanes[1].origin.Z > startingOffsetPlanes[2].origin.Z:
        add (nubsList[6])
    if startingOffsetPlanes[1].origin.Z < startingOffsetPlanes[2].origin.Z:
        add (nubsList[8])
    else:
        add (nubsList[6])
        add (nubsList[8])
    add(overlapSolids[1], mode=Mode.SUBTRACT)
    prunedParts.append(pt.part)

show(prunedParts, reset_camera=Camera.RESET, colors=["#e8b024", "#e8b024", "#e8b024",  "lightblue"])


Too many colors, trimming to length 1
ccccc


# Combining nubs and shafts with throughholes

nubulousShaftsWithThroughHoles = []
iteratorNumbers = []

for i, x in enumerate(shaftListWithThroughHoles):
    with BuildPart() as NubulousShaftWithThroughHole:
        for k, j in enumerate(nubsList):
            iteratorNumber = k // 4
            if iteratorNumber == i and j != 6 and j != 8:
                add(nubsList[k])
            if iteratorNumber == i and j == 6 or iteratorNumber == i and j == 8:
                if startingOffsetPlanes[1].origin.Z > startingOffsetPlanes[2].origin.Z:
                    add (nubsList[6])
                if startingOffsetPlanes[1].origin.Z < startingOffsetPlanes[2].origin.Z:
                    add (nubsList[8])
                else:
                    add (nubsList[6])
                    add (nubsList[8])
            else:
                continue
        add(shaftListWithThroughHoles[i])
        add(overlapSolids, mode = Mode.SUBTRACT)
    nubulousShaftsWithThroughHoles.append(NubulousShaftWithThroughHole.part)

show(nubulousShaftsWithThroughHoles, reset_camera=Camera.RESET)




# Pseudocode
# add the 4 adjacent nubs to their shaft
    # add the shaft i
    # add the 4 nubs whose floor division is equal to i
    # Combine these into single part
# fillet the intersections
# append these combined units to a new list

In [474]:
# type Part = build123d.topology.composite.Part
from build123d.topology.composite import Part

In [475]:
def printPart(part: Part):
    print(part)

printPart(nubsList[0])
printPart(3)

Part at 0x152e70d0710, label(), #children(0)
3


In [476]:
print(type(nubsList[1]))

<class 'build123d.topology.composite.Part'>


In [477]:
# Generating positional arms to connect nubs and basis cylinder

testArms = []
outputArmStream = []
refinedNubsList = [nubsList[1], nubsList[5], nubsList[9], nubsList[13]]

for i, x in enumerate(refinedNubsList):
    arm1, mirrorArm,armTest = armConstructor(i, refinedNubsList[i],startingOffsetPlanes[i])
    outputArmStream.append(arm1)
    outputArmStream.append(mirrorArm)
    testArms.append(armTest)
    # Ensure construction sketches are either nub-centric, or somehow related directionally to the rotation of the arbor centerline. 
show(basisCylinder, shaftListWithThroughHoles, nubsList, outputArmStream, colors=["#e8b024", "#e8b024", "#e8b024",  "lightblue", "pink", "pink"])



Too many colors, trimming to length 4
ccccccccccccccccccccccccccccc


In [478]:
# Pre-fusion chamfers. Use geometry to isolate faces and identify edges as join points between two faces. Merge internal edges this way
input = basisCylinder
# all this will be need to reworked for use with rotational planes and the like. for now it works fine 
xNormalFaces, zNormalFaces = obtainFaces(basisCylinder)
faceList = basisCylinder.faces()

targetVFace = [faceList[9], faceList[11]]
targetZFace = [zNormalFaces[0], zNormalFaces[1]]

vIndicies = [9,3,6,4]
zIndicies = [0,1,1,0]
rads = [2.9,0.9,0.9,2.9]
# Compacted code 
input = basisCylinder
"""
for i, x in enumerate(zIndicies):
    xNormalFaces, zNormalFaces = obtainFaces(input)
    faceList=input.faces()
    targetZFace = [zNormalFaces[0], zNormalFaces[1]]
    show(faceList[vIndicies[i]],targetZFace[zIndicies[i]],input)
    time.sleep(0.5)
    sharedEdges = faceList[vIndicies[i]].edges() & targetZFace[zIndicies[i]].edges()
    input = fillet(sharedEdges, radius = rads[i])
show(input)
"""

show(faceList[3], targetZFace[zIndicies[2]])
# Expanded code. Going to have to just figure out the fillets 1 by 1 with the updated geometries in order to find proper indicies
# for intermediate geometry stages
i = 0
sharedEdges = faceList[vIndicies[i]].edges() & targetZFace[zIndicies[i]].edges()
input = fillet(sharedEdges, radius = rads[i])
show(input)
# Hypothesis: z faces are not changing, but vfaces is 
xNormalFaces, zNormalFaces = obtainFaces(input)
faceList = input.faces()
targetVFace = [faceList[9], faceList[11]]
targetZFace = [zNormalFaces[0], zNormalFaces[1]]
i = i + 1
show(faceList[3], targetZFace[zIndicies[2]])
sharedEdges = faceList[vIndicies[i]].edges() & targetZFace[zIndicies[i]].edges()
input = fillet(sharedEdges, radius = rads[i])
show(input)
# slice 3
xNormalFaces, zNormalFaces = obtainFaces(input)
faceList = input.faces()
targetVFace = [faceList[9], faceList[11]]
targetZFace = [zNormalFaces[0], zNormalFaces[1]]
i = i + 1
show(faceList[6], targetZFace[zIndicies[2]])
sharedEdges = faceList[vIndicies[i]].edges() & targetZFace[zIndicies[i]].edges()
input = fillet(sharedEdges, radius = rads[i])
show(input)
xNormalFaces, zNormalFaces = obtainFaces(input)
faceList = input.faces()
targetVFace = [faceList[9], faceList[11]]
targetZFace = [zNormalFaces[0], zNormalFaces[1]]
i = i + 1
sharedEdges = faceList[vIndicies[i]].edges() & targetZFace[zIndicies[i]].edges()
input = fillet(sharedEdges, radius = rads[i])
show(input)

basisCylinderModified = input

cc
c
cc
c
cc


c
c


In [479]:
i = 0
# Dupicate for testing purposes
# Pre-fusion chamfers. Use geometry to isolate faces and identify edges as join points between two faces. Merge internal edges this way
input = basisCylinder
# all this will be need to reworked for use with rotational planes and the like. for now it works fine 
xNormalFaces, zNormalFaces = obtainFaces(basisCylinder)
faceList = basisCylinder.faces()

targetVFace = [faceList[9], faceList[11]]
targetZFace = [zNormalFaces[0], zNormalFaces[1]]


vIndicies = [9,6,6,8]
zIndicies = [0,0,0,0]
rads = [1,1,6,6]
# Compacted code 
input = basisCylinder
for i, x in enumerate(zIndicies):
    if i == 2 or i == 3:
        xNormalFaces, zNormalFaces = obtainFaces(input)
        faceList=input.faces()
        targetZFace = [zNormalFaces[0], zNormalFaces[1]]
        show(faceList[vIndicies[i]],targetZFace[zIndicies[i]],input)
        time.sleep(0.25)
        sharedEdges = faceList[vIndicies[i]].edges() & targetZFace[zIndicies[i]].edges()
        input = fillet(sharedEdges, radius = rads[i])
    else:
        xNormalFaces, zNormalFaces = obtainFaces(input)
        faceList = input.faces()
        targetZFace = [zNormalFaces[0], zNormalFaces[1]]
        show(faceList[vIndicies[i]], targetZFace[zIndicies[i]])
        time.sleep(0.25)
        sharedEdges = faceList[vIndicies[i]].edges() & targetZFace[zIndicies[i]].edges()
        input = chamfer(sharedEdges, length = 8, length2 = 3.99999, reference = targetZFace[zIndicies[i]])
        show(input)
"""

# Expanded code. Going to have to just figure out the fillets 1 by 1 with the updated geometries in order to find proper indicies
# for intermediate geometry stages
i = 0
show(faceList[vIndicies[i]], targetZFace[zIndicies[i]])
sharedEdges = faceList[vIndicies[i]].edges() & targetZFace[zIndicies[i]].edges()
input = chamfer(sharedEdges, length = 8, length2 = 3.99999, reference = targetZFace[zIndicies[i]])
show(input)
# Hypothesis: z faces are not changing, but vfaces is 
xNormalFaces, zNormalFaces = obtainFaces(input)
faceList = input.faces()
targetZFace = [zNormalFaces[0], zNormalFaces[1]]
i = i + 1
show(faceList[vIndicies[i]], targetZFace[zIndicies[i]])
sharedEdges = faceList[vIndicies[i]].edges() & targetZFace[zIndicies[i]].edges()
input = chamfer(sharedEdges, length = 8, length2 = 3.99999, reference = targetZFace[zIndicies[i]])
show(input)
# slice 3
xNormalFaces, zNormalFaces = obtainFaces(input)
faceList = input.faces()
targetVFace = [faceList[9], faceList[11]]
targetZFace = [zNormalFaces[0], zNormalFaces[1]]
i = i + 1
show(faceList[vIndicies[i]], targetZFace[zIndicies[i]])
sharedEdges = faceList[vIndicies[i]].edges() & targetZFace[zIndicies[i]].edges()
input = fillet(sharedEdges, radius = rads[i])
show(input)
# slice3 4
xNormalFaces, zNormalFaces = obtainFaces(input)
faceList = input.faces()
targetVFace = [faceList[9], faceList[11]]
targetZFace = [zNormalFaces[0], zNormalFaces[1]]
i = i + 1
show(faceList[vIndicies[i]], targetZFace[zIndicies[i]])
sharedEdges = faceList[vIndicies[i]].edges() & targetZFace[zIndicies[i]].edges()
print(sharedEdges)
input = fillet(sharedEdges, radius = rads[i])
show(input)
"""
"""
"""
basisCylinderModified = input
show(basisCylinderModified)

cc
c
cc
c
ccc
ccc
c


In [480]:
# probably make this a function
for i, x in enumerate(faceList):
    show(faceList[i], targetZFace[zIndicies[3]])
    time.sleep(0.25)

cc
cc
cc
cc
-c
cc
cc
cc
cc
cc
cc
cc
cc
cc
cc
cc
cc
cc


In [ ]:
# Constructing final solid part
with BuildPart() as fusedPart:

    # Shafts and side nubs
    for i, x in enumerate(shaftListWithThroughHoles):
        add (shaftListWithThroughHoles[i]), 
        add (nubsList[1::2])
        """
        """
    # separable npx pairs need a union. this is the math for that union.
    add (prunedParts[4])
    if startingOffsetPlanes[1].origin.Z < startingOffsetPlanes[2].origin.Z:
        add (nubsList[6])
    if startingOffsetPlanes[1].origin.Z > startingOffsetPlanes[2].origin.Z:
        add (nubsList[8])
    else:
        add (nubsList[6], nubsList[8])
    
    # paired npx nubs
    add (prunedParts[0])
    add (prunedParts[1])
    add (loft1)

    add (prunedParts[2])
    add (prunedParts[3])
    add (loft2)

with BuildPart() as anotherpart:
    # Basis cylinder
    add (basisCylinderModified)
    add (fusedPart)

    # adding arms
    add (outputArmStream)
    


guideTubeFrameNoChamfer = anotherpart
show(guideTubeFrameNoChamfer, basisCylinderModified, reset_camera=Camera.RESET)

+c


In [494]:
show(fusedPart, reset_camera = Camera.RESET)
with BuildPart() as chamferedPart:
    for i, x in enumerate(shaftListWithThroughHoles):
        filletMe = new_edges( shaftListWithThroughHoles[i], *nubsList[1::2],combined = fusedPart)
    show (chamferedPart)

c


AttributeError: type object 'BuildPart' has no attribute 'cast'

In [482]:
# Post-Fusion chamfers. Use new_edges() to find merged edges from specific parts and sort through these to obtain the edges we care about
